# Brain Tumor Detection, Segmentation & Highlighting

This notebook takes the `dataset/yes` and `dataset/no` brain MRI dataset and does three things end to end:

1. **Classification** — trains a CNN to predict whether an MRI scan shows a tumor (`yes`) or not (`no`).
2. **Segmentation** — since the dataset only has image-level labels (no pixel masks), tumor regions are segmented using classical image-processing (thresholding + morphology + contour analysis) rather than a supervised segmentation model.
3. **Highlighting** — draws the segmented tumor boundary and a bounding box on the original scan, and produces a side-by-side visualization (original / mask / highlighted).

**Expected folder structure** (unzip `dataset.zip` next to this notebook):
```
dataset/
├── yes/   # MRI images that contain a tumor
└── no/    # MRI images with no tumor
```


## 1. Setup & Imports

In [ ]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from glob import glob

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

np.random.seed(42)
tf.random.set_seed(42)

# Path to the extracted dataset folder
DATASET_DIR = "dataset"
YES_DIR = os.path.join(DATASET_DIR, "yes")
NO_DIR = os.path.join(DATASET_DIR, "no")

IMG_SIZE = 128  # images will be resized to IMG_SIZE x IMG_SIZE


## 2. Explore the Dataset

In [ ]:
yes_files = glob(os.path.join(YES_DIR, "*"))
no_files = glob(os.path.join(NO_DIR, "*"))

print(f"Tumor (yes) images : {len(yes_files)}")
print(f"No tumor (no) images: {len(no_files)}")
print(f"Total images        : {len(yes_files) + len(no_files)}")

# Preview a few samples from each class
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, f in enumerate(yes_files[:5]):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
    axes[0, i].imshow(img)
    axes[0, i].set_title("Tumor")
    axes[0, i].axis("off")
for i, f in enumerate(no_files[:5]):
    img = cv2.cvtColor(cv2.imread(f), cv2.COLOR_BGR2RGB)
    axes[1, i].imshow(img)
    axes[1, i].set_title("No Tumor")
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()


## 3. Preprocessing

Each image is:
1. Read and converted to RGB
2. Resized to a fixed size (`IMG_SIZE x IMG_SIZE`)
3. Normalized to `[0, 1]`


In [ ]:
def load_and_preprocess(path, img_size=IMG_SIZE):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (img_size, img_size))
    img = img.astype("float32") / 255.0
    return img

X, y, paths = [], [], []

for f in yes_files:
    X.append(load_and_preprocess(f))
    y.append(1)
    paths.append(f)

for f in no_files:
    X.append(load_and_preprocess(f))
    y.append(0)
    paths.append(f)

X = np.array(X, dtype="float32")
y = np.array(y, dtype="int32")
paths = np.array(paths)

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test, paths_train, paths_test = train_test_split(
    X, y, paths, test_size=0.2, stratify=y, random_state=42
)

print("Train samples:", X_train.shape[0])
print("Test samples :", X_test.shape[0])


## 4. Data Augmentation

The dataset is small (253 images), so augmentation helps the CNN generalize better.


In [ ]:
train_datagen = ImageDataGenerator(
    rotation_range=15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    fill_mode="nearest",
)
train_datagen.fit(X_train)


## 5. Build the CNN Classifier

In [ ]:
def build_model(img_size=IMG_SIZE):
    model = models.Sequential([
        layers.Input(shape=(img_size, img_size, 3)),

        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.Conv2D(256, (3, 3), activation="relu", padding="same"),
        layers.BatchNormalization(),
        layers.MaxPooling2D(2, 2),

        layers.GlobalAveragePooling2D(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.5),
        layers.Dense(1, activation="sigmoid"),
    ])

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss="binary_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model()
model.summary()


## 6. Train the Model

In [ ]:
EPOCHS = 30
BATCH_SIZE = 16

callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
    tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-6),
]

history = model.fit(
    train_datagen.flow(X_train, y_train, batch_size=BATCH_SIZE),
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    callbacks=callbacks,
)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(history.history["accuracy"], label="train")
axes[0].plot(history.history["val_accuracy"], label="val")
axes[0].set_title("Accuracy")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history.history["loss"], label="train")
axes[1].plot(history.history["val_loss"], label="val")
axes[1].set_title("Loss")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


## 7. Evaluate the Classifier

In [ ]:
y_pred_prob = model.predict(X_test).ravel()
y_pred = (y_pred_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=["No Tumor", "Tumor"]))

cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["No Tumor", "Tumor"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


## 8. Tumor Segmentation & Highlighting

The dataset has no pixel-level tumor masks, so segmentation is done with a classical
image-processing pipeline instead of a supervised segmentation network:

1. Convert to grayscale and blur slightly to reduce noise.
2. Apply **Otsu's thresholding** to separate bright regions (tumor tissue is usually
   hyper-intense) from the background/skull.
3. Clean the mask with morphological **opening** and **closing**.
4. Find contours and keep the **largest connected component** (assumed to be the tumor).
5. Draw the tumor's contour and bounding box on the original image, and build a
   red-highlighted overlay.

> Note: this heuristic works well on the classic `yes/no` brain-MRI dataset because the
> tumor tends to be the brightest, most compact region after skull-stripping, but it is
> not a substitute for a clinically validated segmentation model.


In [ ]:
def segment_and_highlight(image_rgb, min_area_ratio=0.002):
    """
    image_rgb : float image in [0, 1], shape (H, W, 3)
    Returns: mask (H, W) uint8, highlighted image (H, W, 3) uint8, contour or None
    """
    img_uint8 = (image_rgb * 255).astype("uint8")
    gray = cv2.cvtColor(img_uint8, cv2.COLOR_RGB2GRAY)

    # Smooth to reduce noise before thresholding
    blurred = cv2.GaussianBlur(gray, (5, 5), 0)

    # Otsu's thresholding to isolate the brightest structures
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Morphological clean-up
    kernel = np.ones((5, 5), np.uint8)
    opened = cv2.morphologyEx(thresh, cv2.MORPH_OPEN, kernel, iterations=2)
    closed = cv2.morphologyEx(opened, cv2.MORPH_CLOSE, kernel, iterations=2)

    # Find contours and keep the largest plausible one (skip contours touching the image border,
    # which are usually skull/background artifacts rather than the tumor)
    contours, _ = cv2.findContours(closed, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    h, w = gray.shape
    min_area = min_area_ratio * h * w
    best_contour = None
    best_area = 0

    for c in contours:
        area = cv2.contourArea(c)
        if area < min_area:
            continue
        x, y, cw, ch = cv2.boundingRect(c)
        touches_border = x <= 1 or y <= 1 or (x + cw) >= w - 1 or (y + ch) >= h - 1
        if touches_border:
            continue
        if area > best_area:
            best_area = area
            best_contour = c

    mask = np.zeros((h, w), dtype="uint8")
    highlighted = img_uint8.copy()

    if best_contour is not None:
        cv2.drawContours(mask, [best_contour], -1, 255, thickness=cv2.FILLED)

        # Red overlay on the tumor region
        overlay = highlighted.copy()
        overlay[mask == 255] = (255, 0, 0)
        highlighted = cv2.addWeighted(overlay, 0.4, highlighted, 0.6, 0)

        # Green contour outline + bounding box
        cv2.drawContours(highlighted, [best_contour], -1, (0, 255, 0), 2)
        x, y, cw, ch = cv2.boundingRect(best_contour)
        cv2.rectangle(highlighted, (x, y), (x + cw, y + ch), (255, 255, 0), 2)

    return mask, highlighted, best_contour


### Visualize segmentation on sample tumor images

In [ ]:
def show_segmentation(path):
    img = load_and_preprocess(path)
    mask, highlighted, contour = segment_and_highlight(img)

    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    axes[0].imshow(img)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(mask, cmap="gray")
    axes[1].set_title("Segmented Mask")
    axes[1].axis("off")

    axes[2].imshow(highlighted)
    axes[2].set_title("Tumor Highlighted")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# Show a few tumor examples from the test set
tumor_test_paths = [p for p, label in zip(paths_test, y_test) if label == 1][:5]
for p in tumor_test_paths:
    show_segmentation(p)


## 9. End-to-End Pipeline

Given a new MRI image path, this function:
1. Classifies it as **Tumor** / **No Tumor** using the trained CNN.
2. If a tumor is predicted, runs the segmentation + highlighting step.
3. Displays the result with the predicted probability.


In [ ]:
def predict_and_highlight(path, threshold=0.5):
    img = load_and_preprocess(path)
    prob = model.predict(img[np.newaxis, ...], verbose=0)[0, 0]
    label = "Tumor" if prob >= threshold else "No Tumor"

    if label == "Tumor":
        mask, highlighted, contour = segment_and_highlight(img)
        display_img = highlighted
        title = f"{label} (p={prob:.2f}) — tumor region highlighted"
    else:
        display_img = (img * 255).astype("uint8")
        title = f"{label} (p={prob:.2f})"

    plt.figure(figsize=(5, 5))
    plt.imshow(display_img)
    plt.title(title)
    plt.axis("off")
    plt.show()

    return label, prob

# Example usage:
# predict_and_highlight("dataset/yes/Y1.jpg")


## 10. Save the Trained Model

In [ ]:
os.makedirs("models", exist_ok=True)
model.save("models/brain_tumor_classifier.h5")
print("Model saved to models/brain_tumor_classifier.h5")


## Summary

- The CNN classifies MRI scans as **Tumor** / **No Tumor**.
- Classical image processing (Otsu thresholding + morphology + contour analysis) segments
  the tumor region on images predicted positive, and the result is overlaid and outlined
  on the original scan.
- `predict_and_highlight(path)` ties both steps together into a single reusable function
  for any new MRI image.

**Possible improvements**
- Use a proper segmentation architecture (U-Net) if pixel-level tumor masks become available.
- Fine-tune a pretrained backbone (e.g. EfficientNet, ResNet) for the classifier.
- Add Grad-CAM to visualize what the CNN focuses on as another form of localization.
